# 🚗 Treinamento do YOLO com o Dataset Oficial UFPR-ALPR (Placas Brasileiras)
### Dataset: UFPR-ALPR (Universidade Federal do Paraná - 4.500 imagens de veículos brasileiros)

Este notebook foi otimizado para:
1. Download Turbo (Multithread) de 9.07 GB em menos de 1 minuto com `aria2c`
2. Conversão e divisão 80% Treino / 20% Validação (com limpeza automática de disco)
3. **Treinamento Otimizado para Memória RAM (sem risco de reinicialização ou crash)**
4. **Download automático imediato do `best.pt`** ao término do treinamento

## 1. Verificação da GPU
Certifique-se de que a GPU está ativada no Colab em: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4**.

In [ ]:
!nvidia-smi

## 2. Instalação das Dependências

In [ ]:
!pip install -q ultralytics pillow
print("✅ Ultralytics instalado com sucesso!")

## 3. Download Turbo (Multithread) e Descompactação do Dataset UFPR-ALPR (9.07 GB)

In [ ]:
import os

ZIP_DESTINO = "/content/UFPR-ALPR.zip"
PASTA_EXTRACAO = "/content/ufpr_raw"

# 1. Instala o acelerador de download aria2
!apt-get install -y aria2 > /dev/null 2>&1

# 2. Baixa com 16 conexões simultâneas de alta velocidade
if not os.path.exists(ZIP_DESTINO):
    print("🚀 Baixando dataset oficial UFPR-ALPR (9.07 GB) em alta velocidade...")
    !aria2c -x 16 -s 16 -j 16 -k 1M "https://www.inf.ufpr.br/vri/databases/yj4Iu2-UFPR-ALPR.zip" -d /content -o UFPR-ALPR.zip
    print("✅ Download concluído com sucesso!")
else:
    print("✅ Arquivo ZIP já presente no ambiente!")

# 3. Descompacta rapidamente com o unzip nativo do Linux
if not os.path.exists(PASTA_EXTRACAO):
    print("📦 Descompactando arquivos (aguarde alguns segundos)...")
    !unzip -q /content/UFPR-ALPR*.zip -d /content/ufpr_raw
    print("✅ Descompactação finalizada com sucesso!")
else:
    print("✅ Pasta descompactada já existe!")

## 4. Conversão das Anotações da UFPR para o Formato YOLO e Limpeza de Disco

In [ ]:
import os
import glob
import shutil
import random
import re
import cv2

DATASET_YOLO = "/content/dataset_ufpr_yolo"

if os.path.exists(DATASET_YOLO):
    shutil.rmtree(DATASET_YOLO)

os.makedirs(os.path.join(DATASET_YOLO, "train/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "train/labels"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "val/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "val/labels"), exist_ok=True)

print("🔍 Buscando anotações nos arquivos do UFPR-ALPR...")
todos_txt = glob.glob("/content/ufpr_raw/**/*.txt", recursive=True)
if not todos_txt:
    todos_txt = glob.glob("/content/**/*.txt", recursive=True)

arquivos_veiculos = [t for t in todos_txt if not os.path.basename(t).lower().startswith("readme") and "dataset_ufpr_yolo" not in t]
print(f"📄 Total de anotações de veículos encontradas: {len(arquivos_veiculos)}")

pares_validos = []
for txt_path in arquivos_veiculos:
    base_nome = txt_path.rsplit('.', 1)[0]
    img_path = None
    for ext in [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"]:
        if os.path.exists(base_nome + ext):
            img_path = base_nome + ext
            break
            
    if not img_path:
        continue
        
    try:
        with open(txt_path, 'r', encoding='latin-1', errors='ignore') as f:
            linhas = f.readlines()
    except Exception:
        continue
        
    boxes_placa = []
    for linha in linhas:
        l = linha.strip()
        l_lower = l.lower()
        
        if "corners" in l_lower or "position_plate" in l_lower:
            nums = [float(n) for n in re.findall(r"[-+]?(?:\d*\.\d+|\d+)", l.split(":")[-1])]
            
            if len(nums) == 8:
                xs = nums[0::2]
                ys = nums[1::2]
                xmin, xmax = min(xs), max(xs)
                ymin, ymax = min(ys), max(ys)
                w = xmax - xmin
                h = ymax - ymin
                if w > 0 and h > 0:
                    boxes_placa.append((xmin, ymin, w, h))
            elif len(nums) >= 4:
                x, y, w, h = nums[:4]
                if w > 0 and h > 0:
                    boxes_placa.append((x, y, w, h))
                    
    if boxes_placa:
        pares_validos.append((img_path, txt_path, boxes_placa))

print(f"✅ Total de {len(pares_validos)} imagens com placas anotadas prontas!")

random.seed(42)
random.shuffle(pares_validos)
corte = int(len(pares_validos) * 0.8)
split_treino = pares_validos[:corte]
split_val = pares_validos[corte:]

def salvar_amostras(amostras, split_nome):
    for img_path, txt_path, boxes in amostras:
        img = cv2.imread(img_path)
        if img is None:
            continue
        h_img, w_img = img.shape[:2]
        
        yolo_labels = []
        for x, y, w, h in boxes:
            x_center = (x + w / 2.0) / w_img
            y_center = (y + h / 2.0) / h_img
            norm_w = w / w_img
            norm_h = h / h_img
            yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
            
        nome_base = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(DATASET_YOLO, f"{split_nome}/images", nome_base))
        
        nome_txt = os.path.basename(img_path).rsplit('.', 1)[0] + ".txt"
        with open(os.path.join(DATASET_YOLO, f"{split_nome}/labels", nome_txt), 'w') as f_out:
            f_out.write("\n".join(yolo_labels))

print("📁 Gravando conjunto de Treino...")
salvar_amostras(split_treino, "train")
print(f"✅ Treino: {len(split_treino)} imagens salvas.")

print("📁 Gravando conjunto de Validação...")
salvar_amostras(split_val, "val")
print(f"✅ Validação: {len(split_val)} imagens salvas.")

# Cria arquivo data.yaml
yaml_content = f"""
path: {DATASET_YOLO}
train: train/images
val: val/images

names:
  0: placa
"""

yaml_path = os.path.join(DATASET_YOLO, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

# Limpa arquivos temporários do ZIP e pasta raw para liberar 34 GB de espaço
print("🧹 Limpando arquivos temporários para liberar memória...")
!rm -rf /content/ufpr_raw /content/UFPR-ALPR*.zip

print(f"\n🎉 Dataset pronto e otimizado para o YOLO: {yaml_path}")

## 5. Treinamento do YOLO (Seguro para RAM, Rápido na GPU T4 e Download Automático)
Configurado com `batch=32`, `workers=4` e sem cache de RAM para garantir 100% de estabilidade sem risco de estouro de memória.

In [ ]:
import os
import gc
from ultralytics import YOLO
from google.colab import files

# Libera qualquer resíduo da memória RAM
gc.collect()

DATASET_YOLO = "/content/dataset_ufpr_yolo"

# Carrega modelo base YOLO
modelo = YOLO("yolov8n.pt")

# Treinamento com GPU T4 e consumo de RAM ultra-baixo
resultados = modelo.train(
    data=os.path.join(DATASET_YOLO, "data.yaml"),
    epochs=40,
    imgsz=640,
    batch=32,          # Tamanho ideal para VRAM e estabilidade
    workers=4,         # Carregamento paralelo balanceado
    amp=True,          # Precisão mista FP16 para máxima velocidade
    patience=10,
    save=True,
    name="yolo_ufpr_placas_brasil"
)

print("🎉 Treinamento com UFPR-ALPR finalizado com sucesso!")

# Download automático imediato do melhor modelo (best.pt)
caminho_pesos = "runs/detect/yolo_ufpr_placas_brasil/weights/best.pt"
if os.path.exists(caminho_pesos):
    print("⬇️ Baixando modelo treinado (best.pt) para o seu computador...")
    files.download(caminho_pesos)
else:
    print("❌ Arquivo best.pt não encontrado.")

## 6. Avaliação dos Gráficos e Métricas

In [ ]:
from IPython.display import Image, display
import glob

for grafico in glob.glob("runs/detect/yolo_ufpr_placas_brasil/*.png"):
    print(f"📊 Gráfico: {grafico}")
    display(Image(filename=grafico))
    print("-" * 50)